# Past Service / Current Service 本次工作整理 v4

這份 Notebook 重新整理本次針對噴塗產線專案中，**Past Service（過去服務）** 與 **Current Service（目前服務）** 所完成的內容。

本版整合了以下重點：

```text
1. v6 欄位命名與 stations（三站）架構
2. sample_method（取樣 / 彙整方法）定義與計算方式
3. metrics（指標資料）/ process_parameters（製程參數）裡的 KPI 與指標欄位
4. alarm_count / defect_count 異常次數保留機制
5. Integration Service / UI Adapter 作為後續統整 Past / Current / Future 的服務
```


# 1. 專案背景與整體流程

本專案是智慧噴塗產線服務架構，包含三個噴塗站：

| station_id | 中文名稱 | 英文名稱 | 製程 |
|---|---|---|---|
| M1 | 底漆站 | Primer Station | 底漆 |
| M2 | 面漆站 | Topcoat Station | 色漆 / 面漆 |
| M3 | 金漆站 | Gold Paint Station | 金漆 |

整體服務流程應整理為：

```text
Edge Service（邊緣服務）
    ↓
Preprocessing（前處理）
    ↓
DB Schema / SQL Search（資料庫結構 / SQL 查詢）
    ↓
Past Service（過去服務）
    ↓
Current Service（目前服務）
    ↓
Future Service（未來預測服務）
    ↓
Integration Service / UI Adapter（整合服務 / UI 轉接層）
    ↓
Statistics Service（統計服務）
    ↓
UI(M) Manager UI（管理層介面）
    ↓
UI(E) Engineer UI（工程師介面）



# 2. 本次工作原則

同學整理的 `service_io_v6` 已經有固定欄位名稱，因此本次不重新命名欄位，而是沿用 v6 架構。

目前主要使用欄位：

```text
stations（站別集合）
station_id（站別編號）
station_name_zh（站別中文名稱）
station_name_en（站別英文名稱）
state（狀態）
metrics（指標資料）
process_parameters（製程參數）
window（視窗）
generated_at（產生時間）
sample_method（取樣 / 彙整方法）
```

舊概念與 v6 欄位對應：

| 原版 | v6 欄位 |
|---|---|
| 3M | stations 裡面的 M1 / M2 / M3 |
| Qc | quality_score_pct（品質分數百分比） |
| Ut | utilization_pct（稼動率） |
| cycle time | cycle_time_sec（週期時間） |
| para | metrics / process_parameters |


# 3. stations（三站）架構

三台機台 / 三個製程站統一放在 `stations（站別集合）` 中。


In [ ]:
STATIONS = [
    {
        "station_id": "M1",
        "station_name_zh": "底漆站",
        "station_name_en": "Primer Station"
    },
    {
        "station_id": "M2",
        "station_name_zh": "面漆站",
        "station_name_en": "Topcoat Station"
    },
    {
        "station_id": "M3",
        "station_name_zh": "金漆站",
        "station_name_en": "Gold Paint Station"
    }
]

STATIONS


# 4. sample_method（取樣 / 彙整方法）定義

`sample_method（取樣 / 彙整方法）` 負責回答：

```text
一段資料進來後，要怎麼取代表值？
```

目前先定義以下幾種 sample_method：

| sample_method | 中文意思 | 算法 | 適合用在哪 |
|---|---|---|---|
| `latest_valid` | 最新有效值 | 取最新一筆非空值 | current state、目前 KPI |
| `mean` | 平均值 | 視窗內有效數值平均 | past 數值型資料 |
| `recent_average` | 最近 N 筆平均 | 最近 N 秒或 N 筆平均 | current pressure、flow、temperature |
| `majority` | 多數決 | 出現最多次 | past state(狀態）但 alarm / defect 另外用 max 或 count 保留下來 |
| `max` | 最大值 | 取最大值 | alarm、clog_rate、risk |
| `count` | 計數 | 次數統計 | alarm_count、defect_count |




# 5. sample_method 計算函式 初版

目前已經在 `past_service.py` 與 `current_service.py` 內加入：

```python
def apply_sample_method(values, method, recent_n=5):
    ...
```



In [ ]:
from collections import Counter

def apply_sample_method(values, method, recent_n=5):
    if values is None:
        return None

    valid_values = [v for v in values if v is not None]

    if not valid_values:
        return None

    if method == "mean":
        numeric_values = [
            v for v in valid_values
            if isinstance(v, (int, float)) and not isinstance(v, bool)
        ]
        if not numeric_values:
            return None
        return sum(numeric_values) / len(numeric_values)

    if method == "latest_valid":
        return valid_values[-1]

    if method == "majority":
        return Counter(valid_values).most_common(1)[0][0]

    if method == "recent_average":
        numeric_values = [
            v for v in valid_values
            if isinstance(v, (int, float)) and not isinstance(v, bool)
        ]
        if not numeric_values:
            return None
        recent_values = numeric_values[-recent_n:]
        return sum(recent_values) / len(recent_values)

    if method == "max":
        numeric_values = [
            v for v in valid_values
            if isinstance(v, (int, float)) and not isinstance(v, bool)
        ]
        if not numeric_values:
            return None
        return max(numeric_values)

    if method == "min":
        numeric_values = [
            v for v in valid_values
            if isinstance(v, (int, float)) and not isinstance(v, bool)
        ]
        if not numeric_values:
            return None
        return min(numeric_values)

    if method == "count":
        # 布林值序列：計算 True 次數，例如 alarm/defect flag
        if all(isinstance(v, bool) for v in valid_values):
            return sum(1 for v in valid_values if v is True)

        # 一般事件序列：計算非 None 的事件數
        return len(valid_values)

    raise ValueError(f"Unsupported sample_method: {method}")


# 6. sample_method 測試

以下示範目前的 sample_method 可以怎麼算。


In [ ]:
state_values = ["Running", "Running", "Standby", "Running", None]
pressure_values = [2.0, 2.1, 2.2, None, 2.3]
temperature_values = [26.0, 26.2, 26.1, 26.4, 26.5]
alarm_flags = [False, False, True, False, True]
defect_flags = [False, True, False, False, False]

print("majority state =", apply_sample_method(state_values, "majority"))
print("latest valid state =", apply_sample_method(state_values, "latest_valid"))
print("mean pressure =", apply_sample_method(pressure_values, "mean"))
print("recent average temperature =", apply_sample_method(temperature_values, "recent_average", recent_n=3))
print("alarm_count =", apply_sample_method(alarm_flags, "count"))
print("defect_count =", apply_sample_method(defect_flags, "count"))


# 7. 為什麼 Past state 用 majority，但還要 alarm_count / defect_count

Past Service（過去服務）看的是一段歷史區間，不是單一當下。

例如某站在過去一段時間的狀態為：

```text
Running
Running
Running
Alarm
Running
Running
```

如果用 `majority（多數決）`：

```text
past_state = Running
```

這代表這段時間的主要狀態是 Running，這是合理的。

但是問題是：

```text
中間發生過一次 Alarm 會被 majority 蓋掉。
```

所以本次新增：

```text
alarm_count（異常次數）
defect_count（缺陷次數）
```

用來保留歷史區間內發生過的異常事件。

最後 output 可以同時表示：

```text
state = Running
alarm_count = 1
defect_count = 0
```

也就是：

```text
主要狀態：Running
但期間發生過 1 次 Alarm
```

這樣 Past Service 的資料就不會把短暫異常完全蓋掉。


# 8. KPI / 指標欄位定義

這些是放在 `metrics（指標資料） / process_parameters（製程參數）` 裡的 KPI 或指標欄位。

| 欄位 | 中文意思 | 用途 |
|---|---|---|
| `quality_score_pct` | 品質分數百分比 | 表示該站品質好壞，例如良率、覆蓋率、厚度合格率 |
| `availability_pct` | 可用率 | 表示機台在計畫時間內有多少比例是可用的 |
| `utilization_pct` | 稼動率 / 使用率 | 表示機台實際運轉時間佔總時間比例 |
| `cycle_time_sec` | 週期時間，單位秒 | 表示一個工件或一批次通過該站花多久 |
| `clog_rate_pct` | 阻塞率 / 堵塞率 | 表示噴嘴、管路或濾網堵塞風險比例 |
| `maintainability_pct` | 維護性指標 | 表示設備維護狀態，通常可理解為越高越好維護、越低越接近需要保養 |
| `alarm_count` | 異常次數 | 表示視窗內發生 Alarm 的次數 |
| `defect_count` | 缺陷次數 | 表示視窗內發生 defect 的次數 |
| `risk_text` | 風險文字 | 例如 Low / Medium / High |

這些欄位對應白板上的概念：

| 白板概念 | 欄位 |
|---|---|
| Quality | quality_score_pct |
| Availability | availability_pct |
| Utilization | utilization_pct |
| Cycle-time | cycle_time_sec |
| Maintainability | maintainability_pct |
| Nozzle / Filter clog risk | clog_rate_pct |
| Alarm / Defect event | alarm_count / defect_count |


# 9. sample_method 與 KPI 公式



## 第一層：KPI 公式

KPI 公式負責產生單筆 KPI 數值。

例如：

```text
utilization_pct（稼動率） = runtime / total_time × 100
cycle_time_sec（週期時間） = part_end_time - part_start_time
quality_score_pct（品質分數百分比） = OK_count（計數） / total_count（計數） × 100
```

這些公式目前還沒有正式定案，需要後續組內討論。

## 第二層：sample_method

sample_method 負責把一段資料做彙整。

例如：

```text
Past Service：
[82, 85, 88, 90] → mean（平均值） → 86.25

Current Service：
[82, 85, 88, 90] → latest_valid（最新有效值） → 90
```

所以目前已完成的是：

```text
sample_method prototype
```

尚未完成的是：

```text
KPI 原始公式
```


# 10. Past Service 的 sample_method 使用方式

Past Service 處理的是歷史資料，因此它偏向「區間統計」。

目前 Past Service 使用：

| 欄位類型 | sample_method（取樣 / 彙整方法） |
|---|---|
| state（狀態） | majority（多數決） |
| pressure_bar | mean（平均值） |
| flow_rate_ml_min | mean（平均值） |
| quality_score_pct（品質分數百分比） | mean（平均值） |
| availability_pct（可用率） | mean（平均值） |
| clog_rate_pct（堵塞率） | mean（平均值） |
| maintainability_pct（維護性指標） | mean（平均值） |
| temperature_c | mean（平均值） |
| utilization_pct（稼動率） | mean（平均值） |
| cycle_time_sec（週期時間） | mean（平均值） |
| risk_text | latest_valid（最新有效值） |

理由：

```text
Past Service 看的是一段歷史範圍。
數值型欄位多數用 mean。
狀態用 majority 找出該時間區間的主要狀態。
Alarm / Defect 另外用 count 保留異常次數。
```


In [ ]:
PAST_SAMPLE_METHOD = {
    "state": "majority",
    "metrics": {
        "pressure_bar": "mean",
        "flow_rate_ml_min": "mean",
        "quality_score_pct": "mean",
        "availability_pct": "mean",
        "clog_rate_pct": "mean",
        "maintainability_pct": "mean",
        "alarm_count": "count",
        "defect_count": "count",
        "risk_text": "latest_valid"
    },
    "process_parameters": {
        "temperature_c": "mean",
        "utilization_pct": "mean",
        "cycle_time_sec": "mean"
    }
}

PAST_SAMPLE_METHOD


# 11. Current Service 的 sample_method 使用方式

Current Service 處理的是目前 / 近即時資料，因此它偏向「最新有效值」與「短視窗」。

目前 Current Service 使用：

| 欄位類型 | sample_method（取樣 / 彙整方法） |
|---|---|
| state（狀態） | latest_valid（最新有效值） |
| pressure_bar | recent_average（最近 N 筆平均） |
| flow_rate_ml_min | recent_average（最近 N 筆平均） |
| temperature_c | recent_average（最近 N 筆平均） |
| quality_score_pct（品質分數百分比） | latest_valid（最新有效值） |
| availability_pct（可用率） | latest_valid（最新有效值） |
| clog_rate_pct（堵塞率） | latest_valid（最新有效值） |
| maintainability_pct（維護性指標） | latest_valid（最新有效值） |
| utilization_pct（稼動率） | latest_valid（最新有效值） |
| cycle_time_sec（週期時間） | latest_valid（最新有效值） |
| risk_text | latest_valid（最新有效值） |

理由：

```text
Current Service 需要反映目前狀態。
state 與 KPI 用 latest_valid。
pressure / flow / temperature 可能會有瞬間跳動，所以用 recent_average 平滑。
alarm_count / defect_count 用 count 顯示目前視窗內的異常事件次數。
```


In [ ]:
CURRENT_SAMPLE_METHOD = {
    "state": "latest_valid",
    "metrics": {
        "pressure_bar": "recent_average",
        "flow_rate_ml_min": "recent_average",
        "quality_score_pct": "latest_valid",
        "availability_pct": "latest_valid",
        "clog_rate_pct": "latest_valid",
        "maintainability_pct": "latest_valid",
        "alarm_count": "count",
        "defect_count": "count",
        "risk_text": "latest_valid"
    },
    "process_parameters": {
        "temperature_c": "recent_average",
        "utilization_pct": "latest_valid",
        "cycle_time_sec": "latest_valid"
    }
}

CURRENT_SAMPLE_METHOD


# 12. Past Service Input / Output 架構

Past Service input 主要包含：

```text
range_variable（歷史查詢範圍）
window（視窗）
sample_method（取樣 / 彙整方法）
target（目標欄位）
stations（三站）
```

Past Service output 主要包含：

```text
generated_at（產生時間）
window（視窗）
sample_method（取樣 / 彙整方法）
stations（三站資料）
```


In [ ]:
past_service_input_template = {
    "service_name": "past_service",
    "schema_version": "v6-compatible",
    "range_variable": {
        "type": "history_range",
        "description": "Past Service 查詢的歷史範圍",
        "start": None,
        "end": None
    },
    "window": {
        "mode": "time_or_batch",
        "window_type": "2hour_or_10batch",
        "window_size": None
    },
    "sample_method": PAST_SAMPLE_METHOD,
    "target": ["state", "metrics", "process_parameters"],
    "stations": STATIONS
}

past_service_input_template


In [ ]:
past_station_output_template = {
    "station_id": "M1",
    "station_name_zh": "底漆站",
    "station_name_en": "Primer Station",
    "state": None,
    "metrics": {
        "pressure_bar": None,
        "flow_rate_ml_min": None,
        "quality_score_pct": None,
        "availability_pct": None,
        "clog_rate_pct": None,
        "maintainability_pct": None,
        "alarm_count": None,
        "defect_count": None,
        "risk_text": None
    },
    "process_parameters": {
        "temperature_c": None,
        "utilization_pct": None,
        "cycle_time_sec": None
    }
}

past_station_output_template


# 13. Current Service Input / Output 架構

Current Service input 主要包含：

```text
window（目前視窗）
sample_method（取樣 / 彙整方法）
target（目標欄位）
stations（三站）
```

Current Service 不需要 `range_variable（歷史查詢範圍）`，因為它處理的是目前或近即時資料。


In [ ]:
current_service_input_template = {
    "service_name": "current_service",
    "schema_version": "v6-compatible",
    "window": {
        "mode": "current",
        "window_type": "current_window",
        "window_size": None,
        "display_label": "current"
    },
    "sample_method": CURRENT_SAMPLE_METHOD,
    "target": ["state", "metrics", "process_parameters"],
    "stations": STATIONS
}

current_service_input_template


# 14. raw_dataset 格式

目前 service 可以接收 `raw_dataset（原始資料集合）` 作為測試資料。

格式如下：


In [ ]:
raw_dataset_example = {
    "M1": {
        "state": ["Running", "Running", "Alarm", "Running"],
        "metrics": {
            "pressure_bar": [2.1, 2.2, 2.0, 2.1],
            "flow_rate_ml_min": [15.0, 15.2, 14.9, 15.1],
            "quality_score_pct": [96.8, 97.0, 95.5, 96.9],
            "availability_pct": [91.0, 92.0, 90.0, 91.5],
            "clog_rate_pct": [1.2, 1.0, 3.5, 1.1],
            "maintainability_pct": [85.0, 86.0, 84.0, 85.5],
            "alarm_count": [False, False, True, False],
            "defect_count": [False, True, False, False],
            "risk_text": ["Low", "Low", "Medium", "Low"]
        },
        "process_parameters": {
            "temperature_c": [26.1, 26.2, 26.5, 26.3],
            "utilization_pct": [88.0, 88.5, 87.0, 88.2],
            "cycle_time_sec": [48.0, 49.0, 50.0, 48.5]
        }
    }
}

raw_dataset_example


# 15. raw_dataset 計算示範

用剛剛的 `raw_dataset_example（原始資料集合範例）` 計算 M1 的 Past Service 結果。


In [ ]:
def calculate_station_output(station, raw_data, sample_method):
    output = {
        "station_id": station["station_id"],
        "station_name_zh": station["station_name_zh"],
        "station_name_en": station["station_name_en"],
        "state": None,
        "metrics": {
            "pressure_bar": None,
            "flow_rate_ml_min": None,
            "quality_score_pct": None,
            "availability_pct": None,
            "clog_rate_pct": None,
            "maintainability_pct": None,
            "alarm_count": None,
            "defect_count": None,
            "risk_text": None
        },
        "process_parameters": {
            "temperature_c": None,
            "utilization_pct": None,
            "cycle_time_sec": None
        }
    }

    output["state"] = apply_sample_method(
        raw_data.get("state"),
        sample_method["state"]
    )

    for field, method in sample_method["metrics"].items():
        output["metrics"][field] = apply_sample_method(
            raw_data.get("metrics", {}).get(field),
            method
        )

    for field, method in sample_method["process_parameters"].items():
        output["process_parameters"][field] = apply_sample_method(
            raw_data.get("process_parameters", {}).get(field),
            method
        )

    return output

m1_past_result = calculate_station_output(
    STATIONS[0],
    raw_dataset_example["M1"],
    PAST_SAMPLE_METHOD
)

m1_past_result


# 16. 本次已完成內容

本次完成：

```text
1. v6 欄位命名對齊
2. stations 三站架構
3. Past Service input / output 初版
4. Current Service input / output 初版
5. sample_method 定義
6. apply_sample_method() 計算函式
7. Past / Current sample_method 對應規則
8. KPI / 指標欄位說明
9. alarm_count / defect_count 異常次數保留機制
10. raw_dataset prototype



# 17. 目前尚未完成內容

目前尚未完成：

```text
1. 真實 DB / SQL Search 串接
2. MQTT / OPC UA / API 串接
3. KPI 原始公式正式定案
4. station state 判斷邏輯
5. Future Service 串接
6. Statistics Service 串接
7. Integration Service 實際 API
8. ontology 補上 SamplingMethod individuals
```


# 18. 本次工作結論

目前這份版本：

```text
Past / Current Service Prototype v1.1
```

已完成：

```text
✔ interface prototype
✔ v6 naming 對齊
✔ station structure
✔ sample_method prototype
✔ sample_method calculation
✔ alarm_count / defect_count
✔ KPI / 指標欄位說明
```

尚未完成：

```text
✘ 真實資料來源
✘ KPI 公式
✘ state logic
✘ Integration Service API 實作
✘ 後續 service / UI 串接
```

目前仍需討論：

```text
1. sample_method 是否合理
2. alarm_count / defect_count 來源是否合理
3. KPI 欄位是否要保留或簡化
4. Current window 要用幾筆或幾秒
```
